<a href="https://colab.research.google.com/github/md-shafiq-cloud-architect/aws-cloud-security-portfolio/blob/main/04-foundational-security-governance/04_foundational_security_governance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re

def validate_lab_baseline(account_id: str, selected_region: str, expected_region: str, enabled_standards: list):
    """Validates AWS Workshop environment baseline and Security Hub CSPM standards."""
    results = {"passed": True, "checks": []}

    # 1. Validate 12-digit AWS Account ID format
    if re.fullmatch(r"\d{12}", account_id):
        results["checks"].append(f"[PASS] Valid 12-digit AWS Account ID: {account_id}")
    else:
        results["passed"] = False
        results["checks"].append(f"[FAIL] Invalid AWS Account ID format: {account_id}")

    # 2. Validate Region Pinning
    if selected_region == expected_region:
        results["checks"].append(f"[PASS] Region pinned accurately: {selected_region}")
    else:
        results["passed"] = False
        results["checks"].append(f"[FAIL] Region mismatch! Selected '{selected_region}', expected '{expected_region}'.")

    # 3. Validate Mandatory Security Hub CSPM Standards
    required_standards = {
        "AWS Foundational Security Best Practices",
        "CIS AWS Foundations Benchmark v1.2.0"
    }
    missing = required_standards - set(enabled_standards)
    if not missing:
        results["checks"].append("[PASS] All mandatory Security Hub CSPM standards are enabled.")
    else:
        results["passed"] = False
        results["checks"].append(f"[FAIL] Missing Security Hub standards: {', '.join(missing)}")

    return results

# Simulated Workshop Environment Check
audit_report = validate_lab_baseline(
    account_id="123456789012",
    selected_region="us-east-1",
    expected_region="us-east-1",
    enabled_standards=[
        "AWS Foundational Security Best Practices",
        "CIS AWS Foundations Benchmark v1.2.0"
    ]
)

for item in audit_report["checks"]:
    print(item)
print(f"\nOverall Baseline Readiness: {'READY' if audit_report['passed'] else 'ACTION REQUIRED'}")

[PASS] Valid 12-digit AWS Account ID: 123456789012
[PASS] Region pinned accurately: us-east-1
[PASS] All mandatory Security Hub CSPM standards are enabled.

Overall Baseline Readiness: READY


In [ ]:
import re

def validate_sns_security_routing(subscriptions: list):
    """
    Validates Amazon SNS topic routing to prevent alert fatigue
    and ensure confirmed human-actionable security notifications.
    """
    allowed_human_topic = "aws-workshop-sns-security-notifications"
    blocked_raw_stream_topic = "aws-workshop-sns-finding-stream"
    email_regex = r"^[\w\.-]+@[\w\.-]+\.\w+$"

    audit_log = []
    all_valid = True

    for sub in subscriptions:
        topic = sub.get("topic_name", "")
        protocol = sub.get("protocol", "").lower()
        endpoint = sub.get("endpoint", "")
        status = sub.get("status", "")

        # Guardrail 1: Block human email subscription on high-volume raw finding stream
        if topic == blocked_raw_stream_topic and protocol == "email":
            all_valid = False
            audit_log.append(
                f"[CRITICAL BLOCK] Email '{endpoint}' subscribed to '{topic}'. "
                f"Risk: ~450+ raw GuardDuty/Security Hub spam alerts (Alert Fatigue)."
            )
            continue

        # Guardrail 2: Verify target notification topic, email format, and confirmation state
        if topic == allowed_human_topic and protocol == "email":
            if not re.match(email_regex, endpoint):
                all_valid = False
                audit_log.append(f"[FAIL] Invalid email endpoint format: {endpoint}")
            elif status != "Confirmed":
                all_valid = False
                audit_log.append(
                    f"[ACTION REQUIRED] Subscription for '{endpoint}' is '{status}'. "
                    f"Click 'Confirm subscription' in inbox."
                )
            else:
                audit_log.append(
                    f"[PASS] Verified active security alert routing: '{topic}' -> '{endpoint}' (Status: {status})"
                )
        else:
            all_valid = False
            audit_log.append(f"[WARN] Unrecognized topic or protocol configuration: {sub}")

    return all_valid, audit_log

# Simulated Lab SNS Configuration Check
sns_test_cases = [
    {
        "topic_name": "aws-workshop-sns-security-notifications",
        "protocol": "email",
        "endpoint": "architect@example.com",
        "status": "Confirmed"
    },
    {
        "topic_name": "aws-workshop-sns-finding-stream",
        "protocol": "email",
        "endpoint": "architect@example.com",
        "status": "PendingConfirmation"
    }
]

is_compliant, logs = validate_sns_security_routing(sns_test_cases)
for entry in logs:
    print(entry)
print(f"\nSNS Alerting Architecture Status: {'SECURE & OPTIMIZED' if is_compliant else 'MISCONFIGURATION DETECTED'}")

[PASS] Verified active security alert routing: 'aws-workshop-sns-security-notifications' -> 'architect@example.com' (Status: Confirmed)
[CRITICAL BLOCK] Email 'architect@example.com' subscribed to 'aws-workshop-sns-finding-stream'. Risk: ~450+ raw GuardDuty/Security Hub spam alerts (Alert Fatigue).

SNS Alerting Architecture Status: MISCONFIGURATION DETECTED


In [ ]:
def evaluate_governance_architecture(trail_cfg: dict, config_cfg: dict, resources: list, alert_chain: list):
    """
    Enterprise Governance & Compliance Validator:
    Audits CloudTrail integrity, AWS Config retention, Config Rule compliance,
    and the 4-link CloudTrail -> CloudWatch -> Alarm -> SNS alert pipeline.
    """
    report = []
    is_compliant = True

    # 1. AWS CloudTrail Multi-Region & Tamper-Evident Validation
    if trail_cfg.get("is_multi_region") and trail_cfg.get("log_file_validation_enabled"):
        report.append("[PASS] CloudTrail: Multi-region active with tamper-evident log validation.")
    else:
        is_compliant = False
        report.append("[FAIL] CloudTrail: Must enable multi-region trail and log file validation.")

    # 2. AWS Config Recorder & 7-Year Retention Check
    if config_cfg.get("recording_enabled") and config_cfg.get("retention_years", 0) >= 7:
        report.append(f"[PASS] AWS Config: Active recorder with {config_cfg['retention_years']}-year history retention.")
    else:
        is_compliant = False
        report.append("[FAIL] AWS Config: Recorder disabled or retention below 7-year compliance baseline.")

    # 3. Detective Compliance Check (Simulating Config Rules - Reports, does NOT block)
    non_compliant_items = [r["id"] for r in resources if not r.get("compliant")]
    if not non_compliant_items:
        report.append("[PASS] Config Rules: All audited resources are COMPLIANT.")
    else:
        is_compliant = False
        report.append(
            f"[DETECTIVE ALERT] Config Rules flagged NON_COMPLIANT resources: {', '.join(non_compliant_items)}. "
            f"(Note: Config Rules report drift; use SCPs for preventive blocking)."
        )

    # 4. 4-Link Security Alert Chain Verification
    expected_chain = ["CloudTrail", "CloudWatchLogs", "MetricFilterAlarm", "AmazonSNS"]
    if alert_chain == expected_chain:
        report.append("[PASS] Alert Pipeline: 4-link chain (CloudTrail -> CloudWatch -> Alarm -> SNS) verified.")
    else:
        is_compliant = False
        report.append(f"[FAIL] Alert Pipeline Broken! Current chain: {' -> '.join(alert_chain)}")

    return is_compliant, report

# Simulated Enterprise Governance Audit
status, audit_findings = evaluate_governance_architecture(
    trail_cfg={"is_multi_region": True, "log_file_validation_enabled": True},
    config_cfg={"recording_enabled": True, "retention_years": 7},
    resources=[
        {"id": "ec2-i-09a8b7c6", "type": "EC2", "compliant": True},
        {"id": "s3-workshop-audit-bucket", "type": "S3", "compliant": False}
    ],
    alert_chain=["CloudTrail", "CloudWatchLogs", "MetricFilterAlarm", "AmazonSNS"]
)

for line in audit_findings:
    print(line)
print(f"\nGovernance & Compliance Posture: {'COMPLIANT' if status else 'REMEDIATION REQUIRED'}")

[PASS] CloudTrail: Multi-region active with tamper-evident log validation.
[PASS] AWS Config: Active recorder with 7-year history retention.
[DETECTIVE ALERT] Config Rules flagged NON_COMPLIANT resources: s3-workshop-audit-bucket. (Note: Config Rules report drift; use SCPs for preventive blocking).
[PASS] Alert Pipeline: 4-link chain (CloudTrail -> CloudWatch -> Alarm -> SNS) verified.

Governance & Compliance Posture: REMEDIATION REQUIRED


In [ ]:
def validate_organizations_governance(org_config: dict):
    """
    Validates AWS Organizations Multi-Account Architecture:
    Checks Management Account isolation, SCP/RCP guardrails,
    Multi-Party Approval quorum, and Non-Prod policy rollout safety.
    """
    findings = []
    is_secure = True

    # 1. Best Practice: Management Account Workload Isolation
    mgmt_workloads = org_config.get("management_account_workloads", [])
    if not mgmt_workloads:
        findings.append("[PASS] Management Account Isolation: Zero application workloads deployed.")
    else:
        is_secure = False
        findings.append(
            f"[CRITICAL FAIL] Management Account contains live workloads: {', '.join(mgmt_workloads)}. "
            f"Move workloads to Member Accounts immediately."
        )

    # 2. Core Preventive Guardrails (SCPs & RCPs)
    required_guardrails = {
        "PreventRootUsageSCP",
        "RestrictRegionsSCP",
        "ProtectSecurityResourcesSCP",
        "RestrictExternalS3AndKMS_RCP",
        "AIServicesOptOutPolicy"
    }
    active_policies = set(org_config.get("active_policies", []))
    missing_policies = required_guardrails - active_policies
    if not missing_policies:
        findings.append("[PASS] SCP/RCP Guardrails: All core preventive & AI opt-out policies active.")
    else:
        is_secure = False
        findings.append(f"[FAIL] Missing Organizations Guardrails: {', '.join(missing_policies)}")

    # 3. Multi-Party Approval Quorum Check (e.g., 2-of-3 within 24h)
    approval = org_config.get("multi_party_approval", {})
    if approval.get("approvals_received", 0) >= approval.get("quorum_required", 2) and approval.get("window_hours", 0) <= 24:
        findings.append(
            f"[PASS] Multi-Party Approval: Quorum met ({approval['approvals_received']}/{approval['total_approvers']}) within 24h window."
        )
    else:
        is_secure = False
        findings.append("[BLOCKED] Multi-Party Approval: Insufficient quorum or 24-hour approval window expired.")

    # 4. Safe Policy Rollout Gate (Non-Prod OU Validation before Prod)
    rollout = org_config.get("policy_rollout", {})
    if rollout.get("target_ou") == "Production" and not rollout.get("tested_in_non_prod"):
        is_secure = False
        findings.append("[CRITICAL BLOCK] Policy Rollout Violation: Cannot push untested SCP/RCP directly to Production OU.")
    else:
        findings.append("[PASS] Policy Rollout Gate: Policy validated in Non-Production OU.")

    return is_secure, findings

# Simulated Multi-Account Organizations Audit
sample_org = {
    "management_account_workloads": [],
    "active_policies": [
        "PreventRootUsageSCP",
        "RestrictRegionsSCP",
        "ProtectSecurityResourcesSCP",
        "RestrictExternalS3AndKMS_RCP",
        "AIServicesOptOutPolicy"
    ],
    "multi_party_approval": {
        "total_approvers": 3,
        "quorum_required": 2,
        "approvals_received": 2,
        "window_hours": 6
    },
    "policy_rollout": {
        "target_ou": "Production",
        "tested_in_non_prod": True
    }
}

passed, audit_results = validate_organizations_governance(sample_org)
for res in audit_results:
    print(res)
print(f"\nAWS Organizations Architecture Status: {'ENTERPRISE READY' if passed else 'ACTION REQUIRED'}")

[PASS] Management Account Isolation: Zero application workloads deployed.
[PASS] SCP/RCP Guardrails: All core preventive & AI opt-out policies active.
[PASS] Multi-Party Approval: Quorum met (2/3) within 24h window.
[PASS] Policy Rollout Gate: Policy validated in Non-Production OU.

AWS Organizations Architecture Status: ENTERPRISE READY


In [ ]:
def validate_control_tower_landing_zone(lz_config: dict, cfn_template_preview: list):
    """
    Validates AWS Control Tower Landing Zone Architecture:
    Checks Shared Accounts (Log Archive & Audit), Mandatory Controls,
    Shift-Left Proactive Controls (CloudFormation Hooks), and Core Integrations.
    """
    audit_log = []
    is_compliant = True

    # 1. Core Shared Accounts Verification
    required_accounts = {"Management", "LogArchive", "Audit"}
    provisioned_accounts = set(lz_config.get("shared_accounts", []))
    missing_accounts = required_accounts - provisioned_accounts
    if not missing_accounts:
        audit_log.append("[PASS] Landing Zone Core: Management, Log Archive, and Audit accounts verified.")
    else:
        is_compliant = False
        audit_log.append(f"[CRITICAL FAIL] Missing Core Shared Accounts: {', '.join(missing_accounts)}")

    # 2. Mandatory Controls Verification (Cannot be disabled)
    mandatory_controls = {
        "DisallowLogArchivePolicyChange",
        "DisallowCloudTrailDisable",
        "EnableRootMFA",
        "DisallowPublicLogArchiveAccess"
    }
    active_controls = set(lz_config.get("enabled_controls", []))
    if mandatory_controls.issubset(active_controls):
        audit_log.append("[PASS] Mandatory Controls: All un-disableable audit & log vault guardrails active.")
    else:
        is_compliant = False
        audit_log.append("[CRITICAL FAIL] Mandatory Control baseline incomplete.")

    # 3. Shift-Left Proactive Control Simulation (CloudFormation Hook - e.g., CT.S3.PR.1)
    proactive_hooks_enabled = lz_config.get("proactive_controls_enabled", False)
    for resource in cfn_template_preview:
        if resource.get("type") == "AWS::S3::Bucket" and resource.get("public_access") is True:
            if proactive_hooks_enabled:
                audit_log.append(
                    f"[PROACTIVE BLOCK - CT.S3.PR.1] CloudFormation Hook blocked '{resource['logical_id']}' "
                    f"pre-deployment: Public S3 access violates Shift-Left Security policy."
                )
            else:
                is_compliant = False
                audit_log.append(
                    f"[FAIL] Public S3 bucket '{resource['logical_id']}' allowed! Enable Proactive Controls."
                )

    # 4. Core Landing Zone Integrations (Identity Center, Backup, Org Trail, Security Hub CSPM)
    required_integrations = {
        "IAMIdentityCenter",
        "AWSBackupCrossAccount",
        "OrganizationCloudTrail",
        "SecurityHubCSPMServiceManaged"
    }
    active_integrations = set(lz_config.get("integrations", []))
    missing_integrations = required_integrations - active_integrations
    if not missing_integrations:
        audit_log.append("[PASS] Landing Zone Integrations: Identity Center, Backup, Org Trail & Security Hub active.")
    else:
        is_compliant = False
        audit_log.append(f"[FAIL] Missing Landing Zone Integrations: {', '.join(missing_integrations)}")

    return is_compliant, audit_log

# Simulated Control Tower Landing Zone & Pre-Deployment Hook Check
landing_zone_state = {
    "shared_accounts": ["Management", "LogArchive", "Audit"],
    "enabled_controls": [
        "DisallowLogArchivePolicyChange",
        "DisallowCloudTrailDisable",
        "EnableRootMFA",
        "DisallowPublicLogArchiveAccess",
        "CT.S3.PR.1"
    ],
    "proactive_controls_enabled": True,
    "integrations": [
        "IAMIdentityCenter",
        "AWSBackupCrossAccount",
        "OrganizationCloudTrail",
        "SecurityHubCSPMServiceManaged"
    ]
}

# Pre-deployment CloudFormation resource simulation
cfn_resources = [
    {"logical_id": "PublicCustomerDataBucket", "type": "AWS::S3::Bucket", "public_access": True}
]

passed, results = validate_control_tower_landing_zone(landing_zone_state, cfn_resources)
for line in results:
    print(line)
print(f"\nControl Tower Landing Zone Posture: {'GOVERNED & SHIFT-LEFT READY' if passed else 'ACTION REQUIRED'}")

[PASS] Landing Zone Core: Management, Log Archive, and Audit accounts verified.
[PASS] Mandatory Controls: All un-disableable audit & log vault guardrails active.
[PROACTIVE BLOCK - CT.S3.PR.1] CloudFormation Hook blocked 'PublicCustomerDataBucket' pre-deployment: Public S3 access violates Shift-Left Security policy.
[PASS] Landing Zone Integrations: Identity Center, Backup, Org Trail & Security Hub active.

Control Tower Landing Zone Posture: GOVERNED & SHIFT-LEFT READY


In [ ]:
def analyze_cloudtrail_forensics(trail_settings: dict, cloudtrail_events: list):
    """
    AWS CloudTrail Forensic & Metric Filter Engine:
    1. Audits trail encryption (SSE-KMS vs SSE-S3 CIS gap) & log validation.
    2. Scans ConsoleLogin events for missing MFA.
    3. Flags high-risk IAM privilege escalation API calls.
    4. Simulates 'ConfigAdministrativeChangesMetricFilter' -> CloudWatch Alarm -> SNS.
    """
    findings = []
    metric_filter_hits = 0

    # 1. Trail Configuration & CIS Benchmark Check
    if trail_settings.get("multi_region") and trail_settings.get("log_file_validation"):
        findings.append("[PASS] CloudTrail Integrity: Multi-region & SHA-256 log validation enabled.")
    if not trail_settings.get("kms_encryption_enabled"):
        findings.append(
            "[CIS BENCHMARK WARN] Log file SSE-KMS encryption is disabled (using SSE-S3 baseline). "
            "Will flag as non-compliant in Security Hub / CIS Benchmark."
        )

    # 2. Forensic Event Stream Analysis
    iam_escalation_apis = {"CreateRole", "AttachRolePolicy", "PutUserPolicy", "CreateAccessKey"}
    config_tamper_apis = {
        "StopConfigurationRecorder", "DeleteConfigurationRecorder",
        "DeleteDeliveryChannel", "DeleteConfigRule", "PutConformancePack"
    }

    for event in cloudtrail_events:
        event_name = event.get("eventName")
        user = event.get("userIdentity", "UnknownUser")
        src_ip = event.get("sourceIPAddress", "0.0.0.0")

        # Check ConsoleLogin without MFA
        if event_name == "ConsoleLogin" and event.get("mfa_used") == "No":
            findings.append(
                f"[SECURITY ALERT] ConsoleLogin WITHOUT MFA detected! User: '{user}', Source IP: {src_ip}"
            )

        # Check IAM Privilege Escalation
        if event_name in iam_escalation_apis:
            findings.append(
                f"[IAM AUDIT] Sensitive IAM API '{event_name}' invoked by '{user}' from {src_ip}."
            )

        # Check ConfigAdministrativeChangesMetricFilter Pattern
        if event_name in config_tamper_apis:
            metric_filter_hits += 1
            findings.append(
                f"[METRIC FILTER +1] '{event_name}' matched ConfigAdministrativeChangesMetricFilter!"
            )

    # 3. CloudWatch Alarm & SNS Trigger State
    alarm_state = "ALARM" if metric_filter_hits >= 1 else "OK"
    sns_action = "Triggered real-time email via SNS Topic" if alarm_state == "ALARM" else "No alert sent"
    findings.append(
        f"[CLOUDWATCH ALARM] Status: {alarm_state} (Metric Count: {metric_filter_hits}) -> {sns_action}"
    )

    return alarm_state, findings

# Simulated CloudTrail Event Log Stream from Workshop Lab
sample_trail_cfg = {
    "multi_region": True,
    "log_file_validation": True,
    "kms_encryption_enabled": False
}

sample_events = [
    {"eventName": "ConsoleLogin", "userIdentity": "dev-engineer", "sourceIPAddress": "198.51.100.42", "mfa_used": "No"},
    {"eventName": "AttachRolePolicy", "userIdentity": "dev-engineer", "sourceIPAddress": "198.51.100.42"},
    {"eventName": "StopConfigurationRecorder", "userIdentity": "suspicious-actor", "sourceIPAddress": "203.0.113.99"}
]

alarm_status, forensic_report = analyze_cloudtrail_forensics(sample_trail_cfg, sample_events)
for entry in forensic_report:
    print(entry)

[PASS] CloudTrail Integrity: Multi-region & SHA-256 log validation enabled.
[CIS BENCHMARK WARN] Log file SSE-KMS encryption is disabled (using SSE-S3 baseline). Will flag as non-compliant in Security Hub / CIS Benchmark.
[SECURITY ALERT] ConsoleLogin WITHOUT MFA detected! User: 'dev-engineer', Source IP: 198.51.100.42
[IAM AUDIT] Sensitive IAM API 'AttachRolePolicy' invoked by 'dev-engineer' from 198.51.100.42.
[METRIC FILTER +1] 'StopConfigurationRecorder' matched ConfigAdministrativeChangesMetricFilter!
[CLOUDWATCH ALARM] Status: ALARM (Metric Count: 1) -> Triggered real-time email via SNS Topic


In [ ]:
def validate_aws_config_monitoring(recorder_cfg: dict, target_instance: dict, change_event: dict, sns_target: dict):
    """
    AWS Config Monitoring & Configuration Item (CI) Diff Engine:
    1. Audits Continuous recording frequency & 7-year S3 history retention.
    2. Enforces Name-Tag isolation to protect production instances.
    3. Captures Before/After configuration state diffs (1-3 min SLA).
    4. Checks SNS configuration streaming for production alert fatigue.
    """
    audit_log = []
    is_valid = True

    # 1. Recorder Frequency & 7-Year S3 Retention Check
    if recorder_cfg.get("recording_on") and recorder_cfg.get("frequency") == "Continuous" and recorder_cfg.get("retention_years", 0) >= 7:
        audit_log.append(
            f"[PASS] Config Recorder: Continuous mode active (1-3 min SLA) -> S3 '{recorder_cfg['s3_bucket']}' (7-yr retention)."
        )
    else:
        is_valid = False
        audit_log.append("[FAIL] Config Recorder: Must use Continuous frequency and 7-year retention.")

    # 2. Resource Tag Isolation Guardrail (Protect non-dev instances)
    allowed_tag = "aws-workshop-instance-dev-server"
    if target_instance.get("name_tag") != allowed_tag:
        is_valid = False
        audit_log.append(
            f"[CRITICAL BLOCK] Unauthorized target '{target_instance.get('id')}' (Tag: {target_instance.get('name_tag')}). "
            f"Only '{allowed_tag}' may be modified."
        )
    else:
        audit_log.append(f"[PASS] Target Verified: '{target_instance['id']}' matches allowed Name tag '{allowed_tag}'.")

    # 3. Configuration Item (CI) Before/After Diff Capture
    old_type = change_event.get("old_instance_type")
    new_type = change_event.get("new_instance_type")
    state_flow = " -> ".join(change_event.get("state_transitions", []))
    audit_log.append(
        f"[CI TIMELINE RECORDED] Instance '{target_instance['id']}' | States: {state_flow} | "
        f"InstanceType Diff: '{old_type}' -> '{new_type}'"
    )

    # 4. Production SNS Configuration Stream Architecture Warning
    if sns_target.get("enabled") and sns_target.get("subscriber_type") == "human_email":
        audit_log.append(
            f"[ARCHITECTURAL WARN] Config changes streaming to '{sns_target['topic']}' (Human Email). "
            f"Lab OK, but in Production route Config streams to SIEM/Lambda to prevent inbox flooding."
        )

    return is_valid, audit_log

# Simulated AWS Config Lab Execution
config_settings = {
    "recording_on": True,
    "frequency": "Continuous",
    "retention_years": 7,
    "s3_bucket": "aws-workshop-logs-bucket"
}

ec2_target = {
    "id": "i-08f7e6d5c4b3a2109",
    "name_tag": "aws-workshop-instance-dev-server"
}

ec2_change = {
    "state_transitions": ["Running", "Stopped", "Running"],
    "old_instance_type": "t3.micro",
    "new_instance_type": "t3.small"
}

sns_stream = {
    "enabled": True,
    "topic": "aws-workshop-sns-security-notifications",
    "subscriber_type": "human_email"
}

ok, logs = validate_aws_config_monitoring(config_settings, ec2_target, ec2_change, sns_stream)
for entry in logs:
    print(entry)
print(f"\nAWS Config Monitoring Status: {'VERIFIED & RECORDING' if ok else 'ACTION REQUIRED'}")

[PASS] Config Recorder: Continuous mode active (1-3 min SLA) -> S3 'aws-workshop-logs-bucket' (7-yr retention).
[PASS] Target Verified: 'i-08f7e6d5c4b3a2109' matches allowed Name tag 'aws-workshop-instance-dev-server'.
[CI TIMELINE RECORDED] Instance 'i-08f7e6d5c4b3a2109' | States: Running -> Stopped -> Running | InstanceType Diff: 't3.micro' -> 't3.small'
[ARCHITECTURAL WARN] Config changes streaming to 'aws-workshop-sns-security-notifications' (Human Email). Lab OK, but in Production route Config streams to SIEM/Lambda to prevent inbox flooding.

AWS Config Monitoring Status: VERIFIED & RECORDING


In [ ]:
def validate_config_rules_closed_loop(active_rules: list, s3_bucket_policy: dict, conformance_pack: dict):
    """
    AWS Config Rules & Closed-Loop Governance Validator:
    1. Audits baseline rules + newly added 's3-bucket-logging-enabled'.
    2. Evaluates S3 bucket policy for 'aws:SecureTransport: false' Deny statement.
    3. Verifies Conformance Pack deployment & closed-loop PutConformancePack -> SNS alert.
    """
    audit_log = []
    all_compliant = True

    # 1. Verify Mandatory + Newly Added Managed Config Rule
    required_rules = {
        "aws-workshop-config-rule-incoming-ssh-disabled",
        "aws-workshop-config-rule-s3-bucket-ssl-requests-only",
        "s3-bucket-logging-enabled"
    }
    missing_rules = required_rules - set(active_rules)
    if not missing_rules:
        audit_log.append(f"[PASS] Config Rules Inventory: All {len(active_rules)} rules active (including s3-bucket-logging-enabled).")
    else:
        all_compliant = False
        audit_log.append(f"[FAIL] Missing Config Rules: {', '.join(missing_rules)}")

    # 2. Evaluate S3 SSL-Only Bucket Policy Remediation
    statements = s3_bucket_policy.get("Statement", [])
    ssl_enforced = any(
        stmt.get("Effect") == "Deny"
        and stmt.get("Action") == "s3:*"
        and stmt.get("Condition", {}).get("Bool", {}).get("aws:SecureTransport") == "false"
        for stmt in statements
    )
    if ssl_enforced:
        audit_log.append("[COMPLIANT] S3 SSL Rule: 'aws:SecureTransport: false' Deny policy verified (HTTP blocked).")
    else:
        all_compliant = False
        audit_log.append("[NON_COMPLIANT] S3 SSL Rule: Bucket allows unencrypted HTTP! Add Deny policy for aws:SecureTransport=false.")

    # 3. Conformance Pack & Closed-Loop CloudTrail -> Alarm -> SNS Trigger
    if conformance_pack.get("deployed") and conformance_pack.get("name") == "s3-best-practices":
        audit_log.append(
            "[CLOSED-LOOP VERIFIED] Conformance Pack 's3-best-practices' deployed -> "
            "CloudTrail logged 'PutConformancePack' -> CloudWatch Alarm [ALARM] -> SNS Email Dispatched!"
        )
    else:
        all_compliant = False
        audit_log.append("[FAIL] Conformance Pack 's3-best-practices' not deployed.")

    return all_compliant, audit_log

# Simulated Closed-Loop Remediation & Conformance Pack Audit
configured_rules = [
    "aws-workshop-config-rule-incoming-ssh-disabled",
    "aws-workshop-config-rule-s3-bucket-ssl-requests-only",
    "s3-bucket-public-read-prohibited",
    "iam-root-access-key-check",
    "encrypted-volumes",
    "ec2-instance-no-public-ip",
    "s3-bucket-logging-enabled"
]

remediated_s3_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "EnforceSSLRequestsOnly",
            "Effect": "Deny",
            "Principal": "*",
            "Action": "s3:*",
            "Resource": "arn:aws:s3:::aws-workshop-test-bucket-123456789012-us-east-1/*",
            "Condition": {"Bool": {"aws:SecureTransport": "false"}}
        }
    ]
}

pack_status = {"deployed": True, "name": "s3-best-practices"}

compliant, report = validate_config_rules_closed_loop(configured_rules, remediated_s3_policy, pack_status)
for line in report:
    print(line)
print(f"\nGovernance & Compliance Domain Status: {'100% COMPLIANT & INTERCONNECTED' if compliant else 'ACTION REQUIRED'}")

[PASS] Config Rules Inventory: All 7 rules active (including s3-bucket-logging-enabled).
[COMPLIANT] S3 SSL Rule: 'aws:SecureTransport: false' Deny policy verified (HTTP blocked).
[CLOSED-LOOP VERIFIED] Conformance Pack 's3-best-practices' deployed -> CloudTrail logged 'PutConformancePack' -> CloudWatch Alarm [ALARM] -> SNS Email Dispatched!

Governance & Compliance Domain Status: 100% COMPLIANT & INTERCONNECTED


In [ ]:
def validate_identity_architecture(workforce_cfg: dict, machine_cfg: dict, app_cfg: dict):
    """
    Enterprise Identity Use-Case & Security Best Practices Validator:
    Audits Workforce (Identity Center, MFA, SAML/OIDC), Machine (IAM Roles, Roles Anywhere, Zero Static Keys),
    and App Identities (Cognito, Verified Permissions Cedar ABAC/RBAC).
    """
    audit_log = []
    is_compliant = True

    # 1. Workforce Identity Validation
    if (
        workforce_cfg.get("service") == "AWS IAM Identity Center"
        and workforce_cfg.get("mfa_enforced")
        and workforce_cfg.get("federation_protocol") in {"SAML", "OIDC"}
    ):
        audit_log.append(
            f"[PASS] Workforce Identity: Centralized SSO via {workforce_cfg['service']} "
            f"({workforce_cfg['federation_protocol']}) with MFA enforced."
        )
    else:
        is_compliant = False
        audit_log.append("[FAIL] Workforce Identity: Must use IAM Identity Center, SAML/OIDC federation, and MFA.")

    # 2. Machine Identity Validation (Zero Hardcoded Secrets & Certificate-based Hybrid Auth)
    if machine_cfg.get("hardcoded_credentials_found"):
        is_compliant = False
        audit_log.append("[CRITICAL FAIL] Machine Identity: Hardcoded access keys detected in code repository!")
    elif machine_cfg.get("cloud_auth") == "IAM Roles" and machine_cfg.get("on_prem_auth") == "IAM Roles Anywhere":
        audit_log.append(
            "[PASS] Machine Identity: Passwordless architecture verified (IAM Roles + IAM Roles Anywhere with PKI/HSM)."
        )
    else:
        is_compliant = False
        audit_log.append("[FAIL] Machine Identity: Use IAM Roles for AWS and IAM Roles Anywhere for hybrid workloads.")

    # 3. App / Customer Identity Validation (Cognito + Verified Permissions)
    if (
        app_cfg.get("authn_service") == "Amazon Cognito"
        and app_cfg.get("authz_engine") == "Amazon Verified Permissions"
        and app_cfg.get("supports_abac_rbac")
    ):
        audit_log.append(
            "[PASS] App Identity: Scalable auth via Amazon Cognito + fine-grained Cedar RBAC/ABAC via Verified Permissions."
        )
    else:
        is_compliant = False
        audit_log.append("[FAIL] App Identity: Configure Amazon Cognito and Amazon Verified Permissions (RBAC/ABAC).")

    return is_compliant, audit_log

# Simulated Enterprise Identity Architecture Audit
workforce_setup = {
    "service": "AWS IAM Identity Center",
    "mfa_enforced": True,
    "federation_protocol": "OIDC"
}

machine_setup = {
    "hardcoded_credentials_found": False,
    "cloud_auth": "IAM Roles",
    "on_prem_auth": "IAM Roles Anywhere"
}

app_setup = {
    "authn_service": "Amazon Cognito",
    "authz_engine": "Amazon Verified Permissions",
    "supports_abac_rbac": True
}

passed, report = validate_identity_architecture(workforce_setup, machine_setup, app_setup)
for entry in report:
    print(entry)
print(f"\nIdentity Architecture Posture: {'ZERO-TRUST READY' if passed else 'REMEDIATION REQUIRED'}")

[PASS] Workforce Identity: Centralized SSO via AWS IAM Identity Center (OIDC) with MFA enforced.
[PASS] Machine Identity: Passwordless architecture verified (IAM Roles + IAM Roles Anywhere with PKI/HSM).
[PASS] App Identity: Scalable auth via Amazon Cognito + fine-grained Cedar RBAC/ABAC via Verified Permissions.

Identity Architecture Posture: ZERO-TRUST READY


In [ ]:
def evaluate_iam_access_request(request_ctx: dict, policies: dict):
    """
    AWS IAM Policy Evaluation Engine & Security Pattern Simulator:
    1. Enforces 'Explicit Deny Always Wins'.
    2. Computes Set Intersection across SCPs, Permission Boundaries, and Identity Policies.
    3. Applies Same-Account Union vs. Cross-Account / KMS 'Both Policies Must Allow' rule.
    4. Validates Conditional MFA and Cross-Account 'sts:ExternalId' (Confused Deputy protection).
    """
    action = request_ctx["action"]
    resource_type = request_ctx["resource_type"]
    is_cross_account = request_ctx.get("is_cross_account", False)

    # Rule 1: Explicit Deny Always Wins
    if action in policies.get("explicit_denies", set()):
        return False, f"[DENIED - Explicit Deny] Action '{action}' is explicitly denied. Explicit Deny always wins!"

    # Security Pattern: Conditional MFA Check
    if policies.get("require_mfa") and not request_ctx.get("mfa_authenticated", False):
        return False, f"[DENIED - Conditional Access] MFA missing ('aws:MultiFactorAuthPresent' != true) for '{action}'."

    # Security Pattern: Confused Deputy Prevention for Cross-Account AssumeRole
    if action == "sts:AssumeRole" and is_cross_account:
        expected_ext_id = policies.get("expected_external_id")
        provided_ext_id = request_ctx.get("external_id")
        if not expected_ext_id or provided_ext_id != expected_ext_id:
            return False, (
                "[DENIED - Confused Deputy Guardrail] Missing or mismatched 'sts:ExternalId' in Trust Policy!"
            )

    # Rule 4: Intersection of Permissions (SCP ∩ Permission Boundary ∩ Identity Policy)
    scp_allowed = policies.get("scp_allowed", set())
    boundary_allowed = policies.get("boundary_allowed", set())
    identity_allowed = policies.get("identity_allowed", set())
    resource_allowed = policies.get("resource_allowed", set())

    effective_identity_perms = scp_allowed & boundary_allowed & identity_allowed

    # Rule 3: Cross-Account or KMS Key requires BOTH Identity & Resource policies to allow
    if is_cross_account or resource_type == "KMS":
        if action in effective_identity_perms and action in resource_allowed:
            return True, (
                f"[ALLOWED - Both Policies Verified] '{action}' on {resource_type} passed Intersection "
                f"(SCP ∩ Boundary ∩ Identity) AND Resource Policy."
            )
        return False, (
            f"[DENIED - Both Policies Must Allow] Cross-account or KMS access for '{action}' requires both "
            f"effective identity permissions and explicit resource policy allow."
        )

    # Same-Account Non-KMS Union Rule (Allowed if in effective identity intersection OR resource policy within SCP)
    if action in effective_identity_perms or (action in resource_allowed and action in scp_allowed):
        return True, (
            f"[ALLOWED - Intersection & Union Verified] '{action}' permitted within SCP ∩ Boundary ∩ Identity limits."
        )

    # Rule 2: Default (Implicit) Deny
    return False, f"[DENIED - Default Implicit Deny] No matching Allow permission found for '{action}'."

# Test Case 1: Set Intersection (Identity allows all, Boundary allows S3+EC2, SCP allows only S3)
sample_policies = {
    "explicit_denies": {"s3:DeleteBucket"},
    "scp_allowed": {"s3:GetObject", "s3:PutObject", "sts:AssumeRole"},
    "boundary_allowed": {"s3:GetObject", "ec2:RunInstances", "sts:AssumeRole"},
    "identity_allowed": {"s3:GetObject", "s3:DeleteBucket", "ec2:RunInstances", "sts:AssumeRole"},
    "resource_allowed": {"s3:GetObject", "sts:AssumeRole"},
    "require_mfa": True,
    "expected_external_id": "ext-org-998877"
}

test_requests = [
    {"action": "s3:GetObject", "resource_type": "S3", "mfa_authenticated": True},
    {"action": "ec2:RunInstances", "resource_type": "EC2", "mfa_authenticated": True},
    {"action": "s3:DeleteBucket", "resource_type": "S3", "mfa_authenticated": True},
    {"action": "sts:AssumeRole", "resource_type": "IAM", "is_cross_account": True, "mfa_authenticated": True, "external_id": "wrong-id"}
]

for req in test_requests:
    allowed, reason = evaluate_iam_access_request(req, sample_policies)
    print(f"Request [{req['action']}] -> {reason}")

Request [s3:GetObject] -> [ALLOWED - Intersection & Union Verified] 's3:GetObject' permitted within SCP ∩ Boundary ∩ Identity limits.
Request [ec2:RunInstances] -> [DENIED - Default Implicit Deny] No matching Allow permission found for 'ec2:RunInstances'.
Request [s3:DeleteBucket] -> [DENIED - Explicit Deny] Action 's3:DeleteBucket' is explicitly denied. Explicit Deny always wins!
Request [sts:AssumeRole] -> [DENIED - Confused Deputy Guardrail] Missing or mismatched 'sts:ExternalId' in Trust Policy!


In [ ]:
def simulate_policy_reduction_and_s3_abac(identity_perms: set, boundary_perms: set, scp_perms: set, s3_policy_rule: dict):
    """
    1. Demonstrates Multi-Layer Permission Reduction (Identity ∩ Boundary ∩ SCP).
    2. Audits S3 ABAC Tag Condition Keys to prevent 'Silent Denial'
       (s3:ExistingObjectTag for GetObject vs s3:RequestObjectTag for PutObject).
    """
    report = []

    # Part 1: Multi-Layer Permission Reduction (Most Restrictive Wins)
    after_boundary = identity_perms & boundary_perms
    final_effective = after_boundary & scp_perms

    report.append(f"[STEP 1 - Identity Baseline] : {sorted(identity_perms)}")
    report.append(f"[STEP 2 - After Boundary]    : {sorted(after_boundary)} (Lambda & EC2 write stripped)")
    report.append(f"[STEP 3 - Final Effective]   : {sorted(final_effective)} (After SCP intersection)")

    # Part 2: S3 ABAC Tag Condition Key Guardrail (Preventing Silent Denial)
    action = s3_policy_rule.get("action")
    condition_key = s3_policy_rule.get("condition_key", "")

    if action == "s3:GetObject" and not condition_key.startswith("s3:ExistingObjectTag/"):
        report.append(
            f"[SILENT DENIAL RISK] Action '{action}' uses '{condition_key}'. "
            f"Must use 's3:ExistingObjectTag/<key>' for read operations!"
        )
    elif action == "s3:PutObject" and not condition_key.startswith("s3:RequestObjectTag/"):
        report.append(
            f"[SILENT DENIAL RISK] Action '{action}' uses '{condition_key}'. "
            f"Must use 's3:RequestObjectTag/<key>' for write operations!"
        )
    else:
        report.append(
            f"[PASS - ABAC Verified] Action '{action}' properly paired with '{condition_key}' (Zero Silent Denial)."
        )

    return report

# Simulated Workshop Calculation & ABAC Check
id_perms = {"s3:GetObject", "s3:PutObject", "s3:DeleteObject", "ec2:DescribeInstances", "ec2:RunInstances", "lambda:InvokeFunction"}
pb_perms = {"s3:GetObject", "s3:PutObject", "s3:DeleteObject", "ec2:DescribeInstances"}
scp_perms = {"s3:GetObject", "s3:ListBucket", "ec2:DescribeInstances"}

# Test a misconfigured PutObject rule using ExistingObjectTag (triggers Silent Denial alert)
abac_rule_test = {
    "action": "s3:PutObject",
    "condition_key": "s3:ExistingObjectTag/Department"
}

for line in simulate_policy_reduction_and_s3_abac(id_perms, pb_perms, scp_perms, abac_rule_test):
    print(line)

[STEP 1 - Identity Baseline] : ['ec2:DescribeInstances', 'ec2:RunInstances', 'lambda:InvokeFunction', 's3:DeleteObject', 's3:GetObject', 's3:PutObject']
[STEP 2 - After Boundary]    : ['ec2:DescribeInstances', 's3:DeleteObject', 's3:GetObject', 's3:PutObject'] (Lambda & EC2 write stripped)
[STEP 3 - Final Effective]   : ['ec2:DescribeInstances', 's3:GetObject'] (After SCP intersection)
[SILENT DENIAL RISK] Action 's3:PutObject' uses 's3:ExistingObjectTag/Department'. Must use 's3:RequestObjectTag/<key>' for write operations!


In [ ]:
import ipaddress
from datetime import datetime, timezone

def evaluate_custom_conditional_policies(req: dict):
    """
    AWS IAM Global Condition Keys & Custom Conditional Policy Engine:
    Simulates WorkshopEC2CloudWatchReadPolicy, WorkshopNetworkRestrictedPolicy,
    WorkshopMFARequiredPolicy (with Explicit Deny), and WorkshopBusinessHoursPolicy.
    """
    results = []
    overall_allowed = True

    # 1. Region Lock Check (aws:RequestedRegion == us-west-2)
    if req.get("requested_region") == "us-west-2":
        results.append("[PASS - Region Lock] aws:RequestedRegion matches 'us-west-2'.")
    else:
        overall_allowed = False
        results.append(f"[DENIED - Region Lock] Region '{req.get('requested_region')}' blocked (only 'us-west-2' allowed).")

    # 2. Corporate Network CIDR Check (IpAddress: aws:SourceIp in 203.0.113.0/24)
    allowed_cidr = ipaddress.ip_network("203.0.113.0/24")
    source_ip = ipaddress.ip_address(req.get("source_ip", "0.0.0.0"))
    if source_ip in allowed_cidr:
        results.append(f"[PASS - Network Boundary] aws:SourceIp ({source_ip}) is within corporate CIDR {allowed_cidr}.")
    else:
        overall_allowed = False
        results.append(f"[DENIED - Network Boundary] Untrusted IP {source_ip} outside {allowed_cidr}!")

    # 3. Defense-in-Depth MFA Guardrail (Explicit Deny if aws:MultiFactorAuthPresent == false)
    if req.get("action") in {"ec2:StopInstances", "ec2:TerminateInstances"}:
        if req.get("mfa_present") is True:
            results.append("[PASS - MFA Guardrail] aws:MultiFactorAuthPresent=true verified for destructive EC2 action.")
        else:
            overall_allowed = False
            results.append("[EXPLICIT DENY - MFA Guardrail] aws:MultiFactorAuthPresent=false! Destructive action blocked.")

    # 4. Business Hours Window Check (DateGreaterThan / DateLessThan on aws:CurrentTime)
    req_time = datetime.fromisoformat(req["current_time"].replace("Z", "+00:00"))
    window_start = datetime(2026, 11, 20, 9, 0, tzinfo=timezone.utc)
    window_end = datetime(2026, 11, 20, 18, 0, tzinfo=timezone.utc)
    if window_start <= req_time <= window_end:
        results.append(f"[PASS - Business Hours] aws:CurrentTime ({req['current_time']}) within maintenance window.")
    else:
        overall_allowed = False
        results.append(f"[DENIED - Business Hours] Request at {req['current_time']} outside allowed window (09:00-18:00 UTC).")

    return overall_allowed, results

# Simulated API Request Testing All 4 Custom Conditional Policies
sample_api_request = {
    "action": "ec2:TerminateInstances",
    "requested_region": "us-west-2",
    "source_ip": "203.0.113.45",
    "mfa_present": False,  # Triggers Defense-in-Depth Explicit Deny
    "current_time": "2026-11-20T14:30:00Z"
}

allowed, audit_logs = evaluate_custom_conditional_policies(sample_api_request)
for log in audit_logs:
    print(log)
print(f"\nFinal Conditional Access Decision: {'PERMITTED' if allowed else 'BLOCKED BY CONDITION GUARDRAIL'}")

[PASS - Region Lock] aws:RequestedRegion matches 'us-west-2'.
[PASS - Network Boundary] aws:SourceIp (203.0.113.45) is within corporate CIDR 203.0.113.0/24.
[EXPLICIT DENY - MFA Guardrail] aws:MultiFactorAuthPresent=false! Destructive action blocked.
[PASS - Business Hours] aws:CurrentTime (2026-11-20T14:30:00Z) within maintenance window.

Final Conditional Access Decision: BLOCKED BY CONDITION GUARDRAIL


In [ ]:
def validate_iam_role_least_privilege(role_cfg: dict, s3_upload_request: dict):
    """
    AWS IAM Roles: Journey to Least Privilege & S3 Defense-in-Depth Validator:
    1. Verifies replacement of 'AmazonS3FullAccess' with scoped 'WorkshopS3MinimalPolicy' (98% reduction).
    2. Checks 1-to-1 lifecycle coupling for sensitive 'RoleSpecificDeleteAccess' inline policy.
    3. Enforces S3 Bucket Policy: mandatory 'AES256' encryption header & PrincipalTag 'Environment=production' for /production/*.
    """
    audit_log = []
    is_secure = True

    # 1. Verify Minimal Policy Scope (No broad AmazonS3FullAccess)
    attached_policies = set(role_cfg.get("managed_policies", []))
    if "AmazonS3FullAccess" in attached_policies:
        is_secure = False
        audit_log.append("[CRITICAL FAIL] Over-privileged 'AmazonS3FullAccess' still attached! Replace with minimal policy.")
    elif "WorkshopS3MinimalPolicy" in attached_policies:
        audit_log.append("[PASS - 98% Scope Reduction] 'WorkshopS3MinimalPolicy' active (ListAllMyBuckets, ListBucket, GetObject, PutObject).")
    else:
        is_secure = False
        audit_log.append("[FAIL] Missing 'WorkshopS3MinimalPolicy'.")

    # 2. Verify Sensitive Delete Permission Lifecycle Coupling (Inline Policy)
    inline_policies = set(role_cfg.get("inline_policies", []))
    if "RoleSpecificDeleteAccess" in inline_policies:
        audit_log.append("[PASS - Lifecycle Coupled] 'RoleSpecificDeleteAccess' (s3:DeleteObject) bound 1-to-1 with IAM Role.")
    else:
        audit_log.append("[INFO] No inline delete policy attached (strict read/write only).")

    # 3. S3 Bucket Policy Defense-in-Depth Evaluation
    target_key = s3_upload_request.get("object_key", "")
    sse_header = s3_upload_request.get("sse_header")
    principal_tags = role_cfg.get("principal_tags", {})

    # 3a. Encryption Header Requirement (s3:x-amz-server-side-encryption == AES256)
    if sse_header != "AES256":
        is_secure = False
        audit_log.append(
            f"[BUCKET POLICY DENY - Encryption] Upload of '{target_key}' rejected! "
            f"Missing mandatory header 's3:x-amz-server-side-encryption: AES256'."
        )
    else:
        audit_log.append(f"[PASS - Encryption Verified] Header 'AES256' confirmed for '{target_key}'.")

    # 3b. Production Folder Principal Tag Restriction (/production/* requires Environment=production)
    if target_key.startswith("production/"):
        if principal_tags.get("Environment") == "production":
            audit_log.append(f"[PASS - Folder Access] PrincipalTag 'Environment=production' verified for '{target_key}'.")
        else:
            is_secure = False
            audit_log.append(
                f"[BUCKET POLICY DENY - Folder Restriction] Role lacks 'Environment=production' PrincipalTag! "
                f"Access to '{target_key}' blocked (use 'dev/' path instead)."
            )
    else:
        audit_log.append(f"[SILENT SUCCESS - Dev Path] Non-production path '{target_key}' permitted within role scope.")

    return is_secure, audit_log

# Simulated Workshop Lab Execution (Testing Encryption + Unauthorized /production/ Upload)
ec2_role_state = {
    "managed_policies": ["WorkshopS3MinimalPolicy"],
    "inline_policies": ["RoleSpecificDeleteAccess"],
    "principal_tags": {}  # Missing Environment=production tag
}

upload_attempt = {
    "object_key": "production/test.txt",
    "sse_header": "AES256"
}

passed, findings = validate_iam_role_least_privilege(ec2_role_state, upload_attempt)
for line in findings:
    print(line)
print(f"\nDefense-in-Depth Request Outcome: {'ALLOWED' if passed else 'BLOCKED BY S3 BUCKET POLICY'}")

[PASS - 98% Scope Reduction] 'WorkshopS3MinimalPolicy' active (ListAllMyBuckets, ListBucket, GetObject, PutObject).
[PASS - Lifecycle Coupled] 'RoleSpecificDeleteAccess' (s3:DeleteObject) bound 1-to-1 with IAM Role.
[PASS - Encryption Verified] Header 'AES256' confirmed for 'production/test.txt'.
[BUCKET POLICY DENY - Folder Restriction] Role lacks 'Environment=production' PrincipalTag! Access to 'production/test.txt' blocked (use 'dev/' path instead).

Defense-in-Depth Request Outcome: BLOCKED BY S3 BUCKET POLICY


In [ ]:
def run_access_analyzer_and_data_protection_audit(lambda_policy: dict, role_usage: dict, data_protection_cfg: dict):
    """
    1. Simulates IAM Access Analyzer Tri-Engine:
       - External Access Analyzer (detects Principal: '*' on Lambda)
       - Unused Access Analyzer (flags unused DynamoDB/SQS full access & unused S3 actions)
       - Internal Access Analyzer (flags over-privileged internal roles on critical S3 bucket)
    2. Audits 4-Tier AWS Data Protection Architecture (KMS FIPS 140-3 L3, Secrets Manager, ACM/Private CA, Macie).
    """
    findings = []
    all_secure = True

    # 1. External Access Analyzer Check (Zone of Trust)
    if lambda_policy.get("Principal") == "*":
        all_secure = False
        findings.append(
            "[EXTERNAL ACCESS FINDING] Lambda resource-based policy allows Principal='*' outside Zone of Trust! "
            "Action: Delete public statement and re-scan to mark Resolved."
        )
    else:
        findings.append("[PASS - External Analyzer] Zero unintended external access outside Zone of Trust.")

    # 2. Unused Access Analyzer Check (90-day CloudTrail Activity vs Granted Permissions)
    granted = set(role_usage.get("granted_permissions", []))
    used = set(role_usage.get("used_in_90_days", []))
    unused_perms = sorted(granted - used)
    if unused_perms:
        all_secure = False
        findings.append(
            f"[UNUSED ACCESS FINDING] Role '{role_usage['role_name']}' holds unused permissions: {unused_perms}. "
            f"Action: Strip unused services/actions to enforce Least Privilege."
        )
    else:
        findings.append(f"[PASS - Unused Analyzer] Role '{role_usage['role_name']}' matches actual 90-day usage.")

    # 3. Internal Access Analyzer Check (Access Path Visibility on Critical S3 Bucket)
    if "s3:*" in granted:
        all_secure = False
        findings.append(
            f"[INTERNAL ACCESS INSIGHT] Over-privileged role '{role_usage['role_name']}' has wildcard 's3:*' "
            f"path to critical bucket. Action: Document justification or scope down to Read-Only."
        )

    # 4. 4-Tier Data Protection Baseline Check
    required_dp_services = {"AWS KMS (FIPS 140-3 L3)", "AWS Secrets Manager", "ACM & Private CA", "Amazon Macie"}
    active_dp = set(data_protection_cfg.get("enabled_services", []))
    missing_dp = required_dp_services - active_dp
    if not missing_dp:
        findings.append("[PASS - Data Protection] All 4 tiers (KMS, Secrets Manager, ACM/Private CA, Macie) verified.")
    else:
        all_secure = False
        findings.append(f"[FAIL - Data Protection] Missing services: {', '.join(missing_dp)}")

    return all_secure, findings

# Simulated Lab Execution
lambda_res_policy = {"Effect": "Allow", "Principal": "*", "Action": "lambda:InvokeFunction"}
demo_role_telemetry = {
    "role_name": "aws-workshop-role-unused-access-demo",
    "granted_permissions": ["s3:*", "dynamodb:*", "sqs:*"],
    "used_in_90_days": ["s3:ListBucket", "s3:GetObject"]
}
dp_baseline = {
    "enabled_services": ["AWS KMS (FIPS 140-3 L3)", "AWS Secrets Manager", "ACM & Private CA", "Amazon Macie"]
}

secure, report = run_access_analyzer_and_data_protection_audit(lambda_res_policy, demo_role_telemetry, dp_baseline)
for line in report:
    print(line)
print(f"\nIAM Governance & Data Protection Status: {'HARDENED' if secure else 'FINDINGS REQUIRE REMEDIATION'}")

[EXTERNAL ACCESS FINDING] Lambda resource-based policy allows Principal='*' outside Zone of Trust! Action: Delete public statement and re-scan to mark Resolved.
[UNUSED ACCESS FINDING] Role 'aws-workshop-role-unused-access-demo' holds unused permissions: ['dynamodb:*', 's3:*', 'sqs:*']. Action: Strip unused services/actions to enforce Least Privilege.
[INTERNAL ACCESS INSIGHT] Over-privileged role 'aws-workshop-role-unused-access-demo' has wildcard 's3:*' path to critical bucket. Action: Document justification or scope down to Read-Only.
[PASS - Data Protection] All 4 tiers (KMS, Secrets Manager, ACM/Private CA, Macie) verified.

IAM Governance & Data Protection Status: FINDINGS REQUIRE REMEDIATION


In [ ]:
def validate_kms_and_s3_encryption(kms_key_policy: dict, rotation_days: int, s3_op: dict):
    """
    AWS KMS Cryptographic Guardrail & S3 SSE-KMS Enforcement Validator:
    1. Enforces KMS Key Policy dominance & Separation of Duties (Encrypt vs Decrypt).
    2. Simulates S3 'DenyObjectsThatAreNotSSEKMS' bucket policy check.
    3. Audits flexible Automatic Key Rotation schedule (90 to 2560 days).
    """
    audit_log = []
    is_compliant = True

    allowed_kms_actions = set(kms_key_policy.get("allowed_actions", []))
    expected_key_arn = kms_key_policy.get("key_arn")

    # 1. Flexible Automatic Key Rotation Check (90 - 2560 days)
    if kms_key_policy.get("auto_rotation_enabled") and 90 <= rotation_days <= 2560:
        audit_log.append(f"[PASS - Key Rotation] Automatic rotation active on {rotation_days}-day schedule (90-2560d window).")
    else:
        is_compliant = False
        audit_log.append("[FAIL - Key Rotation] Enable automatic key rotation within 90 to 2560 days.")

    # 2. S3 Upload Enforcement: DenyObjectsThatAreNotSSEKMS
    if s3_op.get("operation") == "PutObject":
        if s3_op.get("sse_algorithm") != "aws:kms" or s3_op.get("kms_key_arn") != expected_key_arn:
            is_compliant = False
            audit_log.append(
                "[BUCKET POLICY DENY - DenyObjectsThatAreNotSSEKMS] Upload blocked! "
                "Must specify 'aws:kms' and exact 'workshop-encryption-key' ARN."
            )
        elif not {"kms:Encrypt", "kms:GenerateDataKey"}.issubset(allowed_kms_actions):
            is_compliant = False
            audit_log.append("[KMS AccessDeniedException] Principal lacks kms:Encrypt / kms:GenerateDataKey in Key Policy.")
        else:
            audit_log.append("[PASS - SSE-KMS Upload] Object encrypted using Customer Managed KMS Key.")

    # 3. S3 Download / Cryptographic Kill-Switch Check (Requires kms:Decrypt in Key Policy)
    elif s3_op.get("operation") == "GetObject":
        if "kms:Decrypt" not in allowed_kms_actions:
            is_compliant = False
            audit_log.append(
                "[KMS AccessDeniedException - Separation of Duties] S3 GetObject blocked! "
                "S3 IAM permission exists, but KMS Key Policy lacks 'kms:Decrypt' (Cryptographic Kill-Switch active)."
            )
        else:
            audit_log.append("[PASS - SSE-KMS Decrypt] 'kms:Decrypt' verified in Key Policy; object decrypted.")

    return is_compliant, audit_log

# Simulated Workshop KMS Lab Execution (Testing Cryptographic Kill-Switch during S3 Download)
kms_state = {
    "key_arn": "arn:aws:kms:us-east-1:123456789012:key/workshop-encryption-key",
    "allowed_actions": ["kms:Encrypt", "kms:GenerateDataKey"],  # kms:Decrypt intentionally removed
    "auto_rotation_enabled": True
}

s3_download_attempt = {
    "operation": "GetObject",
    "sse_algorithm": "aws:kms",
    "kms_key_arn": "arn:aws:kms:us-east-1:123456789012:key/workshop-encryption-key"
}

passed, report = validate_kms_and_s3_encryption(kms_state, rotation_days=365, s3_op=s3_download_attempt)
for line in report:
    print(line)
print(f"\nKMS Cryptographic Enforcement Status: {'AUTHORIZED' if passed else 'BLOCKED BY KEY POLICY'}")

[PASS - Key Rotation] Automatic rotation active on 365-day schedule (90-2560d window).
[KMS AccessDeniedException - Separation of Duties] S3 GetObject blocked! S3 IAM permission exists, but KMS Key Policy lacks 'kms:Decrypt' (Cryptographic Kill-Switch active).

KMS Cryptographic Enforcement Status: BLOCKED BY KEY POLICY


In [ ]:
def validate_secrets_manager_architecture(lambda_env_vars: dict, secret_cfg: dict, cloudtrail_events: list):
    """
    AWS Secrets Manager Security & Migration Validator:
    1. Scans Lambda environment variables for banned plaintext DB credentials.
    2. Verifies 'USE_SECRETS_MANAGER=true' & Customer Managed KMS Key encryption.
    3. Audits Automatic Rotation strategy (Zero-Downtime Managed Rotation or <=90d Lambda rotation).
    4. Confirms CloudTrail 'GetSecretValue' audit visibility.
    """
    audit_log = []
    is_secure = True

    # 1. Plaintext Environment Variable Exposure Check
    banned_plaintext_keys = {"DB_PASSWORD", "DB_USER", "DB_HOST", "API_SECRET_KEY"}
    leaked_keys = banned_plaintext_keys.intersection(lambda_env_vars.keys())
    if leaked_keys:
        is_secure = False
        audit_log.append(
            f"[CRITICAL LEAK] Plaintext credentials found in Lambda Environment Variables: {sorted(leaked_keys)}. "
            f"Purge immediately after switching to Secrets Manager!"
        )
    else:
        audit_log.append("[PASS - Zero Plaintext] Lambda environment variables purged of hardcoded DB secrets.")

    # 2. Runtime Feature Flag & Custom KMS Encryption Check
    if lambda_env_vars.get("USE_SECRETS_MANAGER") == "true" and secret_cfg.get("kms_key_alias") == "workshop-encryption-key":
        audit_log.append(
            f"[PASS - Encrypted Vault] Runtime switched to Secrets Manager ('{secret_cfg['secret_name']}') "
            f"encrypted with CMK '{secret_cfg['kms_key_alias']}'."
        )
    else:
        is_secure = False
        audit_log.append("[FAIL - Vault Config] Enable USE_SECRETS_MANAGER='true' and attach Customer Managed KMS Key.")

    # 3. Automatic Rotation Strategy Check (Managed vs 90-day Lambda Rotation)
    rotation_mode = secret_cfg.get("rotation_mode")
    rotation_days = secret_cfg.get("rotation_interval_days", 999)
    if rotation_mode == "ManagedRotation" or (rotation_mode == "LambdaRotation" and rotation_days <= 90):
        audit_log.append(
            f"[PASS - Auto Rotation] '{rotation_mode}' active (Zero-downtime / <=90-day compliance verified)."
        )
    else:
        is_secure = False
        audit_log.append("[FAIL - Rotation] Configure Managed Rotation (RDS) or <=90-day Lambda rotation.")

    # 4. CloudTrail GetSecretValue Audit Trail Check
    if "GetSecretValue" in cloudtrail_events:
        audit_log.append("[PASS - CloudTrail Audit] 'GetSecretValue' API retrieval logged for forensic accountability.")
    else:
        is_secure = False
        audit_log.append("[WARN - Audit Gap] No 'GetSecretValue' event recorded in CloudTrail.")

    return is_secure, audit_log

# Simulated Hardened Lambda & Secrets Manager State (Post-Migration)
hardened_lambda_env = {
    "USE_SECRETS_MANAGER": "true",
    "SECRET_NAME": "workshop/database/credentials-123456789012"
}

rds_secret_config = {
    "secret_name": "workshop/database/credentials-123456789012",
    "kms_key_alias": "workshop-encryption-key",
    "rotation_mode": "ManagedRotation",
    "rotation_interval_days": 30
}

trail_logs = ["GetSecretValue", "DescribeSecret"]

passed, report = validate_secrets_manager_architecture(hardened_lambda_env, rds_secret_config, trail_logs)
for line in report:
    print(line)
print(f"\nCredential Storage Posture: {'ZERO-TRUST & ROTATION READY' if passed else 'ACTION REQUIRED'}")

[PASS - Zero Plaintext] Lambda environment variables purged of hardcoded DB secrets.
[PASS - Encrypted Vault] Runtime switched to Secrets Manager ('workshop/database/credentials-123456789012') encrypted with CMK 'workshop-encryption-key'.
[PASS - Auto Rotation] 'ManagedRotation' active (Zero-downtime / <=90-day compliance verified).
[PASS - CloudTrail Audit] 'GetSecretValue' API retrieval logged for forensic accountability.

Credential Storage Posture: ZERO-TRUST & ROTATION READY


In [ ]:
def validate_private_ca_and_alb_tls(ca_cfg: dict, alb_listener: dict, cert_cfg: dict):
    """
    AWS Private CA & ACM TLS Architecture Validator:
    1. Verifies Active Private CA (SHA256 RSA) & Offline Root / Online Intermediate CA best practice.
    2. Checks ALB HTTPS Port 443 listener & domain/wildcard SAN coverage (workshop.internal, *.workshop.internal).
    3. Simulates OpenSSL SNI Chain-of-Trust verification (Subject -> Issuer).
    4. Audits 60-day automated renewal window & short-lived (<=90 days) production cert policy.
    """
    audit_log = []
    is_secure = True

    # 1. Private CA State & Two-Tier PKI Hierarchy Check
    if ca_cfg.get("status") == "ACTIVE" and ca_cfg.get("signing_algorithm") == "SHA256RSA":
        audit_log.append(f"[PASS - Private CA] '{ca_cfg['ca_name']}' is ACTIVE ({ca_cfg['signing_algorithm']}).")
    else:
        is_secure = False
        audit_log.append("[FAIL - Private CA] Root/Subordinate CA must be ACTIVE with SHA256RSA.")

    if ca_cfg.get("hierarchy_pattern") == "OfflineRoot_OnlineIntermediate":
        audit_log.append("[PASS - PKI Hierarchy] Two-tier Offline Root CA + Online Intermediate CA verified.")
    else:
        audit_log.append("[ARCHITECTURAL WARN] Use Offline Root CA + Online Intermediate CA in production.")

    # 2. ALB HTTPS Port 443 Listener & ACM Certificate Attachment
    required_sans = {"workshop.internal", "*.workshop.internal"}
    cert_sans = set(cert_cfg.get("sans", []))
    if alb_listener.get("protocol") == "HTTPS" and alb_listener.get("port") == 443 and required_sans.issubset(cert_sans):
        audit_log.append(
            f"[PASS - ALB TLS 443] HTTPS listener active on Port 443 with SANs {sorted(cert_sans)} ({cert_cfg['key_algorithm']})."
        )
    else:
        is_secure = False
        audit_log.append("[FAIL - ALB Listener] Configure HTTPS Port 443 with workshop.internal & *.workshop.internal SANs.")

    # 3. Simulated OpenSSL SNI Chain-of-Trust Check
    if cert_cfg.get("subject") == "workshop.internal" and cert_cfg.get("issuer") == ca_cfg.get("ca_name"):
        audit_log.append(
            f"[OPENSSL CHAIN VERIFIED] SNI 'workshop.internal' -> Issued by '{cert_cfg['issuer']}' (Data-in-Transit Encrypted)."
        )
    else:
        is_secure = False
        audit_log.append("[FAIL - Chain of Trust] Certificate subject/issuer mismatch.")

    # 4. 60-Day Automated Renewal & Short-Lived Certificate Governance
    if cert_cfg.get("auto_renewal_days_before_expiry") == 60 and cert_cfg.get("validity_days", 999) <= 90:
        audit_log.append("[PASS - Lifecycle] 60-day ACM auto-renewal active with short-lived <=90-day cert validity.")
    else:
        audit_log.append("[INFO - Lifecycle] ACM initiates renewal 60 days prior to expiry; keep prod cert validity <=90 days.")

    return is_secure, audit_log

# Simulated Workshop Private CA & ALB TLS Execution
private_ca_state = {
    "ca_name": "Workshop Private CA",
    "status": "ACTIVE",
    "signing_algorithm": "SHA256RSA",
    "hierarchy_pattern": "OfflineRoot_OnlineIntermediate"
}

alb_443_config = {
    "protocol": "HTTPS",
    "port": 443
}

acm_private_cert = {
    "subject": "workshop.internal",
    "issuer": "Workshop Private CA",
    "sans": ["workshop.internal", "*.workshop.internal"],
    "key_algorithm": "RSA_2048",
    "auto_renewal_days_before_expiry": 60,
    "validity_days": 90
}

passed, report = validate_private_ca_and_alb_tls(private_ca_state, alb_443_config, acm_private_cert)
for line in report:
    print(line)
print(f"\nData-in-Transit & Private PKI Posture: {'ENCRYPTED & TRUST-VERIFIED' if passed else 'ACTION REQUIRED'}")

[PASS - Private CA] 'Workshop Private CA' is ACTIVE (SHA256RSA).
[PASS - PKI Hierarchy] Two-tier Offline Root CA + Online Intermediate CA verified.
[PASS - ALB TLS 443] HTTPS listener active on Port 443 with SANs ['*.workshop.internal', 'workshop.internal'] (RSA_2048).
[OPENSSL CHAIN VERIFIED] SNI 'workshop.internal' -> Issued by 'Workshop Private CA' (Data-in-Transit Encrypted).
[PASS - Lifecycle] 60-day ACM auto-renewal active with short-lived <=90-day cert validity.

Data-in-Transit & Private PKI Posture: ENCRYPTED & TRUST-VERIFIED


In [ ]:
def simulate_detection_and_response_pipeline(guardduty_finding: dict, cspcm_checks: dict, automation_cfg: dict):
    """
    AWS Detection & Self-Healing Response Validator:
    1. Evaluates GuardDuty threat finding severity & log telemetry sources.
    2. Computes Security Hub CSPM Posture Score (0-100 scale across FSBP/CIS/PCI/NIST).
    3. Triggers EventBridge -> Lambda & AWS Config Self-Healing Remediation (MTTR < 10s).
    4. Verifies OCSF Security Lake & Incident Manager readiness.
    """
    audit_log = []

    # 1. GuardDuty Threat Detection & Log Sources Check
    required_sources = {"CloudTrail", "VPCFlowLogs", "DNSLogs"}
    active_sources = set(guardduty_finding.get("telemetry_sources", []))
    if required_sources.issubset(active_sources):
        audit_log.append(
            f"[GUARDDUTY ALERT - Severity {guardduty_finding['severity']}] Detected '{guardduty_finding['type']}' "
            f"across CloudTrail, VPC Flow Logs & DNS Logs."
        )
    else:
        audit_log.append("[FAIL - GuardDuty] Missing mandatory telemetry log sources.")

    # 2. Security Hub CSPM Posture Scoring (0-100 scale)
    passed_controls = cspcm_checks.get("passed", 0)
    total_controls = cspcm_checks.get("total", 200)
    posture_score = round((passed_controls / total_controls) * 100, 1)
    audit_log.append(
        f"[SECURITY HUB CSPM] Posture Score: {posture_score}/100 ({passed_controls}/{total_controls} FSBP/CIS/PCI/NIST checks passed)."
    )

    # 3. EventBridge -> Lambda & AWS Config Self-Healing Remediation
    if automation_cfg.get("eventbridge_active") and automation_cfg.get("config_auto_remediation"):
        audit_log.append(
            f"[SELF-HEALING EXECUTED] EventBridge routed finding -> Lambda & AWS Config remediated drift automatically "
            f"(MTTR: {automation_cfg.get('mttr_seconds', 5)}s) -> Incident Manager & SNS notified."
        )
    else:
        audit_log.append("[WARN - Manual Response] Enable EventBridge & AWS Config auto-remediation to cut MTTR.")

    # 4. OCSF Security Lake & Observability Check
    if automation_cfg.get("ocsf_security_lake") and automation_cfg.get("observability_stack") == ["CloudWatch", "X-Ray", "DevOpsGuru"]:
        audit_log.append("[PASS - Observability & OCSF] Security Lake (OCSF) + CloudWatch, X-Ray & DevOps Guru active.")

    return posture_score, audit_log

# Simulated Workshop Detection & Response Pipeline
gd_event = {
    "type": "CryptoCurrency:EC2/BitcoinTool.B!DNS",
    "severity": 8.0,
    "telemetry_sources": ["CloudTrail", "VPCFlowLogs", "DNSLogs"]
}

cspm_metrics = {"passed": 184, "total": 200}

response_setup = {
    "eventbridge_active": True,
    "config_auto_remediation": True,
    "mttr_seconds": 4,
    "ocsf_security_lake": True,
    "observability_stack": ["CloudWatch", "X-Ray", "DevOpsGuru"]
}

score, report = simulate_detection_and_response_pipeline(gd_event, cspm_metrics, response_setup)
for line in report:
    print(line)
print(f"\nSOC Detection & Response Readiness: {'AUTONOMOUS & ACTIVE' if score >= 80 else 'NEEDS HARDENING'}")

[GUARDDUTY ALERT - Severity 8.0] Detected 'CryptoCurrency:EC2/BitcoinTool.B!DNS' across CloudTrail, VPC Flow Logs & DNS Logs.
[SECURITY HUB CSPM] Posture Score: 92.0/100 (184/200 FSBP/CIS/PCI/NIST checks passed).
[SELF-HEALING EXECUTED] EventBridge routed finding -> Lambda & AWS Config remediated drift automatically (MTTR: 4s) -> Incident Manager & SNS notified.
[PASS - Observability & OCSF] Security Lake (OCSF) + CloudWatch, X-Ray & DevOps Guru active.

SOC Detection & Response Readiness: AUTONOMOUS & ACTIVE


In [ ]:
def evaluate_multi_method_threat_detection(telemetry: dict, threat_intel_db: dict, exposure_signals: dict):
    """
    AWS Multi-Method Threat Detection & Exposure Correlation Engine:
    1. Pattern Recognition: Detects behavioral login anomalies (impossible travel / odd hour).
    2. Threat Intelligence: Matches outbound traffic against Known Bad IPs & Malicious Domains.
    3. Security Hub Correlation: Combines Open Network Port + Vulnerability into High-Risk Exposure Finding.
    4. Audits 4-Angle Comprehensive Monitoring coverage (Zero Blind Spots).
    """
    findings = []
    risk_detected = False

    # 1. Pattern Recognition (Behavioral Baseline Anomaly)
    if telemetry.get("login_country") != telemetry.get("baseline_country"):
        risk_detected = True
        findings.append(
            f"[BEHAVIORAL ANOMALY] User '{telemetry['user']}' logged in from '{telemetry['login_country']}' "
            f"(Baseline: '{telemetry['baseline_country']}') at odd hour — Unknown Threat alert triggered!"
        )

    # 2. Threat Intelligence Feed Match (Known Bad IPs / Malicious Domains)
    dest_ip = telemetry.get("outbound_ip")
    dest_domain = telemetry.get("dns_query")
    if dest_ip in threat_intel_db.get("bad_ips", set()) or dest_domain in threat_intel_db.get("malicious_domains", set()):
        risk_detected = True
        findings.append(
            f"[THREAT INTEL MATCH] EC2 instance contacted Known Bad IP ({dest_ip}) / Malicious Domain ({dest_domain}) "
            f"— Potential Botnet C2 or Crypto Mining!"
        )

    # 3. Security Hub Automated Correlation (Exposure Finding / Attack Path)
    if exposure_signals.get("public_port_open") and exposure_signals.get("critical_cve_present"):
        risk_detected = True
        findings.append(
            "[SECURITY HUB EXPOSURE FINDING] Correlated signal: Publicly reachable port + Critical Inspector CVE! "
            "Attack Path mapped for immediate remediation."
        )

    # 4. 4-Angle Comprehensive Monitoring Coverage Check
    required_angles = {"Network", "API_Activity", "Resource_State", "User_Behavior"}
    active_angles = set(telemetry.get("monitoring_angles", []))
    if required_angles.issubset(active_angles):
        findings.append("[PASS - Layered Coverage] All 4 monitoring angles active (Zero Blind Spots).")
    else:
        findings.append(f"[WARN - Blind Spot] Missing monitoring angles: {sorted(required_angles - active_angles)}")

    return risk_detected, findings

# Simulated Multi-Method Detection Telemetry
sample_telemetry = {
    "user": "finance-admin",
    "baseline_country": "US",
    "login_country": "RU",
    "outbound_ip": "198.51.100.250",
    "dns_query": "://minexmr.com",
    "monitoring_angles": ["Network", "API_Activity", "Resource_State", "User_Behavior"]
}

intel_feeds = {
    "bad_ips": {"198.51.100.250", "203.0.113.88"},
    "malicious_domains": {"://minexmr.com", "c2-botnet.bad"}
}

hub_correlation = {
    "public_port_open": True,
    "critical_cve_present": True
}

threat_found, report = evaluate_multi_method_threat_detection(sample_telemetry, intel_feeds, hub_correlation)
for line in report:
    print(line)
print(f"\nThreat Detection Engine Verdict: {'ACTIVE THREATS & EXPOSURE CORRELATED' if threat_found else 'CLEAN'}")

[BEHAVIORAL ANOMALY] User 'finance-admin' logged in from 'RU' (Baseline: 'US') at odd hour — Unknown Threat alert triggered!
[THREAT INTEL MATCH] EC2 instance contacted Known Bad IP (198.51.100.250) / Malicious Domain (://minexmr.com) — Potential Botnet C2 or Crypto Mining!
[SECURITY HUB EXPOSURE FINDING] Correlated signal: Publicly reachable port + Critical Inspector CVE! Attack Path mapped for immediate remediation.
[PASS - Layered Coverage] All 4 monitoring angles active (Zero Blind Spots).

Threat Detection Engine Verdict: ACTIVE THREATS & EXPOSURE CORRELATED


# 🛡️ Module 04: Foundational Security, Identity, and Governance

**Chief Cloud Architect:** MD Shafiqul Islam Shraban (`md-shafiq-cloud-architect`)  
**Domain:** Threat Detection, CSPM, Event-Driven Incident Response & Automated Remediation  

---

## 🏗️ Enterprise Security Architecture & Event Flow

This architecture integrates continuous threat detection (**Amazon GuardDuty**), centralized posture management (**AWS Security Hub**), compliance tracking (**AWS Config**), and real-time containment (**EventBridge + AWS Lambda + SSM Automation**).

```mermaid
flowchart TD
    GD[Amazon GuardDuty<br/>Threat Intelligence] -->|Severity >= 4.01| EB[Amazon EventBridge<br/>Event Bus & Pattern Rules]
    SH[AWS Security Hub<br/>CSPM & Exposure Findings] -->|HIGH / CRITICAL| EB
    CFG[AWS Config<br/>Compliance Monitoring] -->|NON_COMPLIANT| EB
    CT[AWS CloudTrail<br/>IAM / S3 Admin Actions] -->|errorCode: false| EB

    EB -->|Severity >= 7.0 & EC2| LMB[AWS Lambda<br/>IsolateCompromisedEC2Instance]
    LMB -->|Zero-Traffic SG + Tag| EC2[(Quarantined EC2<br/>Forensic State)]

    CFG -->|SSH Port 22 Open| SSM1[SSM Automation<br/>DisablePublicAccessForSG]
    CFG -->|S3 SSL Missing| SNS[Amazon SNS<br/>Security Ops Alert]
    EB -->|Filtered Alerts| SNS
```

---

## ⚖️ Automated Remediation vs. Human-in-the-Loop Trade-Offs

| Security Scenario | Trigger Source | Automated Action | Architectural Justification |
| :--- | :--- | :--- | :--- |
| **Unrestricted SSH (`0.0.0.0/0:22`)** | AWS Config Rule | **Auto-Revoke Ingress** via SSM Runbook | Immediate external attack surface; zero legitimate production business impact when revoked. |
| **Compromised EC2 (`Severity >= 7.0`)** | GuardDuty + EventBridge | **Network Quarantine** via Lambda (`isolation-sg`) | Preserves memory/disk state for forensic investigation without terminating the instance. |
| **Missing S3 SSL-Only Policy** | AWS Config Rule | **SNS Alert Only** (Human Approval) | Auto-applying bucket policies can break legacy production pipelines; requires human review. |
| **Low-Severity Dev Findings** | Security Hub Automation | **Auto-Suppress** (`Workflow: Suppressed`) | Eliminates SOC alert fatigue while maintaining a complete compliance audit trail. |

---

## 🔍 Validation & Production Audit Script

- **`security_posture_auditor.py`**: Validates least-privilege IAM trust/inline policies (`iam:PassedToService` enforcement), tests ASFF finding routing logic offline, and supports live AWS account auditing via `boto3`.

In [10]:
import json
import logging
from typing import Any, Dict, List

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("EnterpriseSecurityAuditor")


class EnterpriseSecurityGovernanceAuditor:
    """Production-grade auditor for IAM Remediation Roles, GuardDuty, Security Hub (ASFF), and AWS Config."""

    ALLOWED_REMEDIATION_ACTIONS = {
        "ec2:RevokeSecurityGroupIngress",
        "ec2:RevokeSecurityGroupEgress",
        "sns:Publish",
        "iam:PassRole",
    }
    EXPECTED_TRUST_PRINCIPALS = {"://amazonaws.com", "://amazonaws.com"}

    def audit_iam_remediation_role(
        self, trust_policy: Dict[str, Any], inline_policy: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Validates Trust Policy principals and Least-Privilege inline permissions."""
        findings = []

        # 1. Verify Trust Relationship Principals
        trusted_services = set()
        for stmt in trust_policy.get("Statement", []):
            service = stmt.get("Principal", {}).get("Service", [])
            if isinstance(service, str):
                trusted_services.add(service)
            elif isinstance(service, list):
                trusted_services.update(service)

        missing_principals = self.EXPECTED_TRUST_PRINCIPALS - trusted_services
        if missing_principals:
            findings.append(
                f"CRITICAL: Missing trust principals: {sorted(missing_principals)}"
            )

        # 2. Verify Inline Policy Least-Privilege & PassRole Condition
        for stmt in inline_policy.get("Statement", []):
            if stmt.get("Effect") != "Allow":
                continue
            actions = stmt.get("Action", [])
            if isinstance(actions, str):
                actions = [actions]

            for action in actions:
                if (
                    "*" in action
                    or action not in self.ALLOWED_REMEDIATION_ACTIONS
                ):
                    findings.append(
                        f"HIGH: Over-permissive or unauthorized IAM action '{action}'"
                    )

            if "iam:PassRole" in actions:
                conditions = stmt.get("Condition", {}).get("StringEquals", {})
                if "iam:PassedToService" not in conditions:
                    findings.append(
                        "MEDIUM: 'iam:PassRole' lacks 'iam:PassedToService' condition."
                    )

        return {"compliant": len(findings) == 0, "issues": findings}

    def route_asff_finding(self, asff_finding: Dict[str, Any]) -> str:
        """Evaluates standard AWS Security Finding Format (ASFF) payloads."""
        product = asff_finding.get("ProductName", "")
        normalized_severity = asff_finding.get("Severity", {}).get(
            "Normalized", 0
        )
        record_state = asff_finding.get("RecordState", "ACTIVE")
        config_rule = asff_finding.get("ProductFields", {}).get(
            "aws/config/ConfigRuleName", ""
        )

        if record_state != "ACTIVE":
            return "SKIP_ARCHIVED"

        if product == "GuardDuty":
            return (
                "ESCALATE_SNS_PAGERDUTY"
                if normalized_severity >= 40
                else "LOG_LOW_SEVERITY"
            )

        if product == "Security Hub" and normalized_severity < 40:
            return "AUTO_SUPPRESS_DEV_NOISE"

        if product == "Config":
            if config_rule == "aws-workshop-config-rule-incoming-ssh-disabled":
                return "SSM_AUTO_REMEDIATE_REVOKE_SSH"
            if config_rule == "aws-workshop-config-rule-s3-bucket-ssl-requests-only":
                return "SNS_HUMAN_APPROVAL_REQUIRED"

        return "SOC_MANUAL_TRIAGE"


def run_live_aws_audit() -> None:
    """Connects to live AWS environment via boto3 if credentials are present."""
    try:
        import boto3
        from botocore.exceptions import BotoCoreError, ClientError

        sts = boto3.client("sts")
        account_id = sts.get_caller_identity()["Account"]
        logger.info(
            "Connected to Live AWS Account: %s. Fetching Security Hub findings...",
            account_id,
        )

        sh = boto3.client("securityhub")
        response = sh.get_findings(
            Filters={"RecordState": [{"Value": "ACTIVE", "Comparison": "EQUALS"}]},
            MaxResults=5,
        )
        auditor = EnterpriseSecurityGovernanceAuditor()
        for f in response.get("Findings", []):
            decision = auditor.route_asff_finding(f)
            logger.info("Finding ID: %s -> Action: %s", f.get("Id"), decision)

    except Exception as exc:
        logger.warning(
            "Live AWS mode skipped (running in offline validation mode): %s", exc
        )


if __name__ == "__main__":
    auditor = EnterpriseSecurityGovernanceAuditor()

    # Enterprise-hardened IAM Trust & Inline Policy Verification
    trust_doc = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {
                "Service": ["://amazonaws.com", "://amazonaws.com"]
            },
            "Action": "sts:AssumeRole",
        }],
    }
    inline_doc = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": [
                "ec2:RevokeSecurityGroupIngress",
                "ec2:RevokeSecurityGroupEgress",
                "sns:Publish",
                "iam:PassRole",
            ],
            "Resource": "*",
            "Condition": {
                "StringEquals": {"iam:PassedToService": "://amazonaws.com"}
            },
        }],
    }

    iam_result = auditor.audit_iam_remediation_role(trust_doc, inline_doc)
    assert iam_result["compliant"], f"IAM Audit Failed: {iam_result['issues']}"
    logger.info("IAM Remediation Role Audit: PASSED (Zero Privilege Escalation Risk)")

    # Standard ASFF Payloads Simulation
    asff_samples = [
        {"ProductName": "GuardDuty", "Severity": {"Normalized": 80}},
        {"ProductName": "Security Hub", "Severity": {"Normalized": 20}},
        {
            "ProductName": "Config",
            "ProductFields": {
                "aws/config/ConfigRuleName": (
                    "aws-workshop-config-rule-incoming-ssh-disabled"
                )
            },
        },
        {
            "ProductName": "Config",
            "ProductFields": {
                "aws/config/ConfigRuleName": (
                    "aws-workshop-config-rule-s3-bucket-ssl-requests-only"
                )
            },
        },
    ]

    decisions = [auditor.route_asff_finding(item) for item in asff_samples]
    assert decisions == [
        "ESCALATE_SNS_PAGERDUTY",
        "AUTO_SUPPRESS_DEV_NOISE",
        "SSM_AUTO_REMEDIATE_REVOKE_SSH",
        "SNS_HUMAN_APPROVAL_REQUIRED",
    ]
    logger.info("ASFF Automated Triage & Remediation Routing: PASSED")
    run_live_aws_audit()

In [12]:
def transform_custom_action_event(event: dict) -> str:
    """Simulates Security Hub Custom Action EventBridge Input Transformer & naming rules."""
    detail = event.get("detail", {})
    action_name = detail.get("actionName", "")
    assert (
        len(action_name) <= 20
    ), f"Custom action name '{action_name}' exceeds 20 chars"

    finding = detail.get("findings", [{}])[0]
    finding_id = finding.get("Id", "UNKNOWN")
    title = finding.get("Title", "N/A")
    severity = finding.get("Severity", {}).get("Label", "UNKNOWN")
    resource_id = finding.get("Resources", [{}])[0].get("Id", "N/A")

    return (
        "Security Hub CSPM Custom Action Executed\n"
        f"Action: {action_name} | Severity: {severity}\n"
        f"Finding ID: {finding_id} | Title: {title} | Resource: {resource_id}"
    )


# Verify Topic 29 Custom Action & Input Transformer
sample_custom_event = {
    "source": "aws.securityhub",
    "detail-type": "Security Hub Findings - Custom Action",
    "detail": {
        "actionName": "EscalateCritical",
        "findings": [{
            "Id": "arn:aws:securityhub:us-east-1:123456789012:finding/999",
            "Title": "EC2.19 Security groups should not allow unrestricted access",
            "Severity": {"Label": "CRITICAL"},
            "Resources": [{"Id": "i-0abcd1234efgh5678"}],
        }],
    },
}
formatted_alert = transform_custom_action_event(sample_custom_event)
assert "EscalateCritical" in formatted_alert and "CRITICAL" in formatted_alert
print(
    "Topic 29 Custom Action & Input Transformer Validated:\n" + formatted_alert
)

Topic 29 Custom Action & Input Transformer Validated:
Security Hub CSPM Custom Action Executed
Action: EscalateCritical | Severity: CRITICAL
Finding ID: arn:aws:securityhub:us-east-1:123456789012:finding/999 | Title: EC2.19 Security groups should not allow unrestricted access | Resource: i-0abcd1234efgh5678


In [13]:
import logging
import re
from typing import Any, Dict, List

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("DevSecOpsPipelineGate")


class PipelineDependencySecurityGate:
    """Enterprise CI/CD SBOM & Amazon Inspector Vulnerability Gate Validator."""

    BLOCKING_SEVERITIES = {"CRITICAL", "HIGH"}
    VULNERABLE_PIN_RULES = {
        "urllib3": {
            "min_safe_version": (2, 7, 0),
            "cve": "CVE-2026-21441",
        }
    }

    @staticmethod
    def _parse_version(ver_str: str) -> tuple:
        return tuple(int(part) for part in re.findall(r"\d+", ver_str)[:3])

    def audit_requirements_txt(self, requirements_lines: List[str]) -> Dict[str, Any]:
        """Audits pinned Python dependencies against known high-risk CVE baselines."""
        issues = []
        for line in requirements_lines:
            clean = line.strip()
            if not clean or clean.startswith("#") or "==" not in clean:
                continue
            pkg, ver = clean.split("==", 1)
            pkg = pkg.strip().lower()
            if pkg in self.VULNERABLE_PIN_RULES:
                rule = self.VULNERABLE_PIN_RULES[pkg]
                if self._parse_version(ver) < rule["min_safe_version"]:
                    issues.append(
                        f"BLOCKED: {pkg}=={ver} vulnerable to {rule['cve']} "
                        f"(Upgrade to >={'.'.join(map(str, rule['min_safe_version']))})"
                    )
        return {"passed": len(issues) == 0, "issues": issues}

    def evaluate_inspector_scan_summary(
        self, findings: List[Dict[str, str]]
    ) -> str:
        """Enforces CodeBuild GATE_PASSED or GATE_FAILED based on severity thresholds."""
        blocking_cves = [
            f"{f['cve']} ({f['severity']})"
            for f in findings
            if f.get("severity", "").upper() in self.BLOCKING_SEVERITIES
        ]
        if blocking_cves:
            logger.error("Pipeline Gate Blocked due to: %s", blocking_cves)
            return "GATE_FAILED"
        logger.info(
            "Pipeline Gate Passed (Only LOW/MEDIUM findings within policy tolerance)."
        )
        return "GATE_PASSED"


if __name__ == "__main__":
    gate = PipelineDependencySecurityGate()

    # 1. Test Vulnerable State (Before Remediation)
    vulnerable_reqs = ["flask==3.0.0", "urllib3==1.25.8"]
    pre_audit = gate.audit_requirements_txt(vulnerable_reqs)
    assert not pre_audit["passed"], "Expected vulnerable urllib3 to fail audit"

    scan_before = [
        {"cve": "CVE-2026-21441", "severity": "HIGH"},
        {"cve": "CVE-2024-47081", "severity": "MEDIUM"},
    ]
    assert gate.evaluate_inspector_scan_summary(scan_before) == "GATE_FAILED"

    # 2. Test Remediated State (After GitOps Patch to urllib3==2.7.0)
    patched_reqs = ["flask==3.0.0", "urllib3==2.7.0"]
    post_audit = gate.audit_requirements_txt(patched_reqs)
    assert post_audit["passed"], f"Unexpected failure: {post_audit['issues']}"

    scan_after = [{"cve": "CVE-2024-47081", "severity": "MEDIUM"}]
    assert gate.evaluate_inspector_scan_summary(scan_after) == "GATE_PASSED"

    logger.info(
        "ALL CHECKS PASSED: Topic 32 CI/CD SBOM & Inspector Gate Validated."
    )

ERROR:DevSecOpsPipelineGate:Pipeline Gate Blocked due to: ['CVE-2026-21441 (HIGH)']


In [15]:
import ast
import hashlib
import logging
from typing import Any, Dict, List

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("InspectorCodeAuditor")


class InspectorCodeAndRiskAuditor:
    """Detects CWE-78 (OS Command Injection), CWE-328 (Weak MD5), and scores Inspector risk."""

    def audit_lambda_source_code(self, source_code: str) -> List[Dict[str, Any]]:
        """Static AST check mirroring Amazon Inspector Lambda Code Scanning."""
        findings = []
        tree = ast.parse(source_code)

        for node in ast.walk(tree):
            # Detect CWE-78: os.system() or subprocess with shell=True
            if isinstance(node, ast.Call):
                func_name = ""
                if isinstance(node.func, ast.Attribute):
                    func_name = node.func.attr
                elif isinstance(node.func, ast.Name):
                    func_name = node.func.id

                if func_name == "system":
                    findings.append({
                        "line": node.lineno,
                        "cwe": "CWE-78",
                        "severity": "HIGH",
                        "issue": "OS Command Injection via os.system()",
                    })
                elif func_name == "md5":
                    findings.append({
                        "line": node.lineno,
                        "cwe": "CWE-328",
                        "severity": "HIGH",
                        "issue": (
                            "Weak Password Hashing via MD5 (Use SHA-256/bcrypt)"
                        ),
                    })
        return findings

    def prioritize_inspector_finding(self, finding: Dict[str, Any]) -> str:
        """Evaluates Inspector Risk Score + Network Reachability + Fix Availability."""
        score = finding.get("inspector_score", 0.0)
        reachable = finding.get("network_reachable", False)
        fix_available = finding.get("fix_available", False)

        if score >= 8.0 and reachable and fix_available:
            return "P1_IMMEDIATE_PATCH"
        if score >= 7.0 and fix_available:
            return "P2_SCHEDULED_FIX"
        return "P3_MONITOR"


if __name__ == "__main__":
    auditor = InspectorCodeAndRiskAuditor()

    # 1. Simulate Vulnerable Lambda Code from Lab (MD5 + OS Command Injection)
    vulnerable_lambda = """
import hashlib, os

def hash_password(pwd):
    return hashlib.md5(pwd.encode()).hexdigest()

def run_diag(user_host):
    os.system("ping -c 1 " + user_host)
"""
    code_issues = auditor.audit_lambda_source_code(vulnerable_lambda)
    detected_cwes = {item["cwe"] for item in code_issues}
    assert detected_cwes == {
        "CWE-328",
        "CWE-78",
    }, f"Missed CWEs: {code_issues}"
    logger.info("Static Lambda Code Scan Passed: Detected %s", code_issues)

    # 2. Simulate Contextual Inspector Risk Prioritization
    critical_net_bug = {
        "inspector_score": 9.1,
        "network_reachable": True,
        "fix_available": True,
    }
    assert (
        auditor.prioritize_inspector_finding(critical_net_bug)
        == "P1_IMMEDIATE_PATCH"
    )
    logger.info("ALL CHECKS PASSED: Topic 33 Inspector Code & Risk Validated.")
print("ALL CHECKS PASSED: Detected CWEs ->", detected_cwes)

ALL CHECKS PASSED: Detected CWEs -> {'CWE-328', 'CWE-78'}


In [16]:
import urllib.parse


class AWSWAFSecuritySimulator:
    """Simulates AWS WAF Web ACL evaluation, Count/Block rollout, and URL-decode anti-evasion."""

    def __init__(self, mode: str = "COUNT", protected_path: str = "/prod/admin"):
        self.mode = mode.upper()
        self.protected_path = protected_path

    def evaluate_request(self, raw_uri: str, payload: str = "") -> dict:
        # 1. Text Transformation: URL Decode to prevent encoding bypass (e.g., %61dmin -> admin)
        decoded_uri = urllib.parse.unquote(raw_uri).lower()
        decoded_payload = urllib.parse.unquote(payload).lower()

        matched_rule = None

        # 2. Custom Rule: block-admin-path
        if self.protected_path in decoded_uri:
            matched_rule = "CustomRule:block-admin-path"
        # 3. AWS Managed Rule Groups (SQLi & Core XSS)
        elif any(
            sig in decoded_payload
            for sig in ["union select", "or 1=1", "<script>"]
        ):
            matched_rule = "AWSManagedRules:CoreAndSQLi"

        if not matched_rule:
            return {"status_code": 200, "action": "ALLOW", "rule": None}

        if self.mode == "COUNT":
            return {
                "status_code": 200,
                "action": "COUNT_LOGGED",
                "rule": matched_rule,
            }
        return {
            "status_code": 403,
            "action": "BLOCK_FORBIDDEN",
            "rule": matched_rule,
        }


if __name__ == "__main__":
    # Phase 1: Safe Production Rollout (Count Mode)
    waf_count = AWSWAFSecuritySimulator(mode="COUNT")
    res_count = waf_count.evaluate_request("/prod/items", "id=1 OR 1=1")
    assert (
        res_count["status_code"] == 200 and res_count["action"] == "COUNT_LOGGED"
    )

    # Phase 2: Enforcement Mode (Block Mode) + Anti-Evasion URL Decode Test
    waf_block = AWSWAFSecuritySimulator(mode="BLOCK")
    assert (
        waf_block.evaluate_request("/prod/items", "<script>alert(1)</script>")[
            "status_code"
        ]
        == 403
    )
    assert waf_block.evaluate_request("/prod/admin")["status_code"] == 403

    # Attacker tries URL-encoded bypass (/prod/%61dmin) -> Caught by URL Decode transformation
    evasion_test = waf_block.evaluate_request("/prod/%61dmin")
    assert (
        evasion_test["status_code"] == 403
        and evasion_test["rule"] == "CustomRule:block-admin-path"
    )
    assert waf_block.evaluate_request("/prod/home")["status_code"] == 200

    print(
        "ALL CHECKS PASSED: Topic 34 AWS WAF Count/Block & URL-Decode"
        " Anti-Evasion Validated!"
    )

ALL CHECKS PASSED: Topic 34 AWS WAF Count/Block & URL-Decode Anti-Evasion Validated!
